In [1]:
import torch
from deepsnap.dataset import GraphDataset
from deepsnap.graph import Graph

from src.architectures import HomoGNN
from src.config import PROJECT_ROOT
from src.data import DatasetLoader, GraphBuilder, Preprocessor, InferenceResults
from src.db import PBWarehouse
from src.models import Features

model_path = PROJECT_ROOT / "best_model.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HomoGNN(input_size=5, hidden_size=32).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))

2025-07-31 08:23:37.544 | INFO     | src.config:<module>:27 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-07-31 08:23:37.544 | INFO     | src.config:<module>:64 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-07-31 08:23:37.545 | INFO     | src.config:<module>:65 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data


<All keys matched successfully>

In [2]:
dataset_loader = DatasetLoader(PBWarehouse())
data = dataset_loader.load_dataset()
preprocessor = Preprocessor(data=data)
data = preprocessor.preprocess()

node_features = Features(
    tweet=[
        "favorite_count",
        # "retweet_count",
        # "bookmark_count",
        "reply_count",
        "quote_count",
        # "views",
        "source",
        "is_hateful",
    ],
    user=[
        # "favourites_count",
        # "follower_count",
        # "following_count",
        # "number_of_tweets",
        # "listed_count",
        # "is_blue_verified",
        "friends",
    ],
)

gb = GraphBuilder(data=data, node_features=node_features)

graph = gb.create_graph()


# graph.add_node(
#                 1601883222987735040,
#                 node_label=3,
#                 node_feature=torch.tensor([1233, 0, 0, 0, 0], dtype=torch.float32),
#                 node_type="user",
#             )

dataset = GraphDataset([graph], task="link_pred", edge_train_mode="disjoint")

✅ Loading cached dataset - Done.
✅ Loading dataset - Done.
✅ Categorizing source column - Done.
✅ Preprocessing data - Done.


ValueError: Dataset contains null values. Please clean the data.

In [12]:
model.eval()
with torch.no_grad():
    node_embeddings, edge_label_index = model(dataset[0].to(DEVICE))
    results = InferenceResults(graph=graph, node_embeddings=node_embeddings, edge_label_index=edge_label_index)
    print(node_embeddings)
    print(node_embeddings.shape)
    print(edge_label_index)

    nodes_first = torch.index_select(node_embeddings, 0, edge_label_index[0, :].long())
    nodes_second = torch.index_select(node_embeddings, 0, edge_label_index[1, :].long())
    pred = torch.sum(nodes_first * nodes_second, dim=-1)

    print(pred)
    print(pred.shape)

    # For demonstration purposes, we can print the first few predictions
    print("First 10 predictions:", pred[:10])

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0024, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       device='cuda:0')
torch.Size([60893, 32])
tensor([[    0,     2,     3,  ..., 60859, 60868, 60880],
        [    1,     3, 13238,  ..., 60858, 60867, 60879]], device='cuda:0')
tensor([0.0013, 0.0016, 0.0015,  ..., 0.0153, 0.0011, 0.0012], device='cuda:0')
torch.Size([71834])
First 10 predictions: tensor([0.0013, 0.0016, 0.0015, 0.0015, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012,
        0.0012], device='cuda:0')


In [11]:
dataset[0].to(DEVICE)

Graph(G=[], edge_index=[2, 71834], edge_label_index=[2, 71834], negative_label_val=[1], node_feature=[60893, 5], node_label=[60893], node_label_index=[60893], node_type=[60893], task=[])

In [9]:
user = graph.get_random_user()
user = 80706659
topk_results = results.get_top_k_similar_nodes_linked_to_user(user_id=user, descending=True, label=0)

print(f"Top similar nodes linked to user {user}:")
topk_results

Top similar nodes linked to user 80706659:


[(1259187904657666048, 1.0),
 (1259197603247525890, 1.0),
 (1267376131621126145, 1.0),
 (1272551867004805120, 1.0),
 (1248999520425390080, 0.9290196299552917),
 (1275511506264784898, 0.5191972255706787),
 (1280052547831468032, 0.5),
 (1279574131323760641, 0.5),
 (1280324942597427206, 0.5),
 (1284689816102359045, 0.5)]

In [6]:
results.node_list[0]

(1269916756719685632,
 {'node_label': 1,
  'node_feature': tensor([0., 0., 0., 0., 1.]),
  'node_type': 'tweet'})

In [6]:
graph.get_random_user()

48618576

In [7]:
graph.number_of_edges
edge_label_index.shape

torch.Size([2, 71834])

In [9]:
graph.number_of_edges() * 2

71834

In [53]:
graph.number_of_nodes()

60893

In [57]:
embeddings.ndim

2

In [56]:
edge_label_index

tensor([[    0,     2,     3,  ..., 60859, 60868, 60880],
        [    1,     3, 13238,  ..., 60858, 60867, 60879]])

In [49]:
embeddings

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0024, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])

In [43]:
node_logts = torch.sum(nodes_second * torch.index_select(embeddings, 0, torch.tensor([32571])), dim=-1)
node_pred = torch.sigmoid(node_logts)
node_pred

tensor([0.5003, 0.5003, 0.5003,  ..., 0.5003, 0.5002, 0.5002])

In [44]:
scores = []
for thing in zip(indexes, torch.index_select(node_pred, 0, torch.tensor(indexes))):
    scores.append(thing)

scores = sorted(scores, key=lambda x: x[1], reverse=True)
scores

[(60771, tensor(0.5007)),
 (55483, tensor(0.5007)),
 (1332, tensor(0.5007)),
 (44011, tensor(0.5006)),
 (6518, tensor(0.5006)),
 (44744, tensor(0.5005)),
 (55656, tensor(0.5005)),
 (31591, tensor(0.5005)),
 (17082, tensor(0.5004)),
 (4034, tensor(0.5004)),
 (54075, tensor(0.5004)),
 (46821, tensor(0.5004)),
 (45594, tensor(0.5004)),
 (20517, tensor(0.5004)),
 (37767, tensor(0.5004)),
 (58803, tensor(0.5004)),
 (42960, tensor(0.5004)),
 (25786, tensor(0.5004)),
 (47564, tensor(0.5003)),
 (32393, tensor(0.5003)),
 (35053, tensor(0.5003)),
 (29908, tensor(0.5003)),
 (36467, tensor(0.5003)),
 (42767, tensor(0.5003)),
 (11557, tensor(0.5003)),
 (31358, tensor(0.5003)),
 (40945, tensor(0.5003)),
 (23197, tensor(0.5003)),
 (22025, tensor(0.5003)),
 (47301, tensor(0.5003)),
 (44105, tensor(0.5003)),
 (35509, tensor(0.5003)),
 (50036, tensor(0.5003)),
 (15211, tensor(0.5003)),
 (56725, tensor(0.5003)),
 (22447, tensor(0.5003)),
 (9794, tensor(0.5002)),
 (41836, tensor(0.5002)),
 (46437, tensor(

In [ ]:
get_node_index(graph, 751661942)

In [13]:
torch.sort(node_pred)

torch.return_types.sort(
values=tensor([0.5000, 0.5000, 0.5000,  ..., 0.5050, 0.5054, 0.5054]),
indices=tensor([ 8564, 16191, 27432,  ..., 19202,  4608,  8748]))

In [14]:
edge_label_index

tensor([[    0,     2,     3,  ..., 60859, 60868, 60880],
        [    1,     3, 13238,  ..., 60858, 60867, 60879]])

In [5]:
nodes_first.shape

torch.Size([71834, 32])

In [6]:
nodes_first

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0024, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0006, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.1307, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0023, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0046, 0.0000]])

In [8]:
nodes_second.shape

torch.Size([71834, 32])

In [11]:
nodes_second * nodes_first

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.3730e-06,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.7337e-05,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]])

In [27]:
graph.nodes[1269916756719685632]

{'node_label': 1,
 'node_feature': tensor([0., 0., 0., 0., 1.]),
 'node_type': 'test_node_type'}

In [31]:
graph.adj[23719684]

AtlasView({1277976913743503365: {}, 1271591661873901568: {}, 1271595817091182592: {}})

In [ ]:
# THIS IS HOW TO FIND THE NODE!!
# NEXT FIND THE EDGE BETWEEN TWO NODES

# if we could find the original mapping to the deepsnap graph then we good
for thing in graph.adjacency():
    print(thing)
    break

(1269916756719685632, {1264846057906806786: {}})


In [22]:
G: Graph = dataset[0]

G.node_feature

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [6.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.8600e+03, 3.3400e+02, 2.8000e+01, 0.0000e+00, 1.0000e+00],
        ...,
        [6.4000e+01, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [1.8000e+01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00]])

In [55]:
len(G.node_feature)

60893

In [16]:
graph.nodes[23719684]

{'node_label': 3,
 'node_feature': tensor([1595.,    0.,    0.,    0.,    0.]),
 'node_type': 'test_node_type'}

In [42]:
G.edge_label_index

tensor([[    0,     2,     3,  ..., 60859, 60868, 60880],
        [    1,     3, 13238,  ..., 60858, 60867, 60879]])

In [43]:
G.edge_index

tensor([[    0,     2,     3,  ..., 60859, 60868, 60880],
        [    1,     3, 13238,  ..., 60858, 60867, 60879]])

In [31]:
list(graph.adjacency())[1332]

(1280052547831468032, {1158919762040066048: {}})

In [18]:
len(list(graph.adjacency()))

60893

In [47]:
list(graph.nodes(data=True))

[(1269916756719685632,
  {'node_label': 1,
   'node_feature': tensor([0., 0., 0., 0., 1.]),
   'node_type': 'test_node_type'}),
 (1264846057906806786,
  {'node_label': 3,
   'node_feature': tensor([6., 0., 0., 0., 0.]),
   'node_type': 'test_node_type'}),
 (1277976913743503365,
  {'node_label': 1,
   'node_feature': tensor([4.8600e+03, 3.3400e+02, 2.8000e+01, 0.0000e+00, 1.0000e+00]),
   'node_type': 'test_node_type'}),
 (23719684,
  {'node_label': 3,
   'node_feature': tensor([1595.,    0.,    0.,    0.,    0.]),
   'node_type': 'test_node_type'}),
 (1285714950896443395,
  {'node_label': 1,
   'node_feature': tensor([2., 0., 0., 0., 1.]),
   'node_type': 'test_node_type'}),
 (1081973336010309632,
  {'node_label': 3,
   'node_feature': tensor([427.,   0.,   0.,   0.,   0.]),
   'node_type': 'test_node_type'}),
 (1284209535561818112,
  {'node_label': 1,
   'node_feature': tensor([4., 1., 0., 0., 1.]),
   'node_type': 'test_node_type'}),
 (190751291,
  {'node_label': 3,
   'node_feature'

In [48]:
hashmap = {i: node for i, node in enumerate(list(graph.nodes(data=True)))}
hashmap

{0: (1269916756719685632,
  {'node_label': 1,
   'node_feature': tensor([0., 0., 0., 0., 1.]),
   'node_type': 'test_node_type'}),
 1: (1264846057906806786,
  {'node_label': 3,
   'node_feature': tensor([6., 0., 0., 0., 0.]),
   'node_type': 'test_node_type'}),
 2: (1277976913743503365,
  {'node_label': 1,
   'node_feature': tensor([4.8600e+03, 3.3400e+02, 2.8000e+01, 0.0000e+00, 1.0000e+00]),
   'node_type': 'test_node_type'}),
 3: (23719684,
  {'node_label': 3,
   'node_feature': tensor([1595.,    0.,    0.,    0.,    0.]),
   'node_type': 'test_node_type'}),
 4: (1285714950896443395,
  {'node_label': 1,
   'node_feature': tensor([2., 0., 0., 0., 1.]),
   'node_type': 'test_node_type'}),
 5: (1081973336010309632,
  {'node_label': 3,
   'node_feature': tensor([427.,   0.,   0.,   0.,   0.]),
   'node_type': 'test_node_type'}),
 6: (1284209535561818112,
  {'node_label': 1,
   'node_feature': tensor([4., 1., 0., 0., 1.]),
   'node_type': 'test_node_type'}),
 7: (190751291,
  {'node_labe

In [46]:
def get_node_using_node_index(graph: Graph, node_index: int):
    """
    Get the node using its index in the graph.
    """
    nodes = list(graph.nodes(data=True))
    return nodes[node_index]
get_node_using_node_index(graph, 60771)

(1281649502244438016,
 {'node_label': 0,
  'node_feature': tensor([0., 0., 0., 0., 0.]),
  'node_type': 'test_node_type'})

In [ ]:
def get_node_index_using_node(graph: Graph, node_id: int) -> int:
    """
    Get the index of a node in the graph by its ID.
    
    Args:
        graph (Graph): The graph object.
        node_id (int): The ID of the node.
        
    Returns:
        int: The index of the node in the graph.
    """
    nodes = list(graph.nodes(data=True))
    for index, (node, data) in enumerate(nodes):
        if node == node_id:
            return index
        
get_node_index(graph, 751661942)

32571

In [7]:
def get_index_of_nodes_with_label(graph: Graph, label: int):
    return [i for i, x in enumerate(list(graph.nodes(data=True))) if x[1]["node_label"] == label]

In [21]:
indexes = get_index_of_nodes_with_label(graph, 0)
indexes

[1332,
 2305,
 4034,
 6518,
 7009,
 9794,
 11557,
 12726,
 13191,
 15211,
 17082,
 20517,
 21473,
 22025,
 22447,
 23197,
 23371,
 25786,
 29908,
 31358,
 31591,
 32393,
 35053,
 35509,
 36467,
 36968,
 37767,
 38191,
 38577,
 40945,
 41836,
 42767,
 42960,
 44011,
 44105,
 44744,
 44887,
 45594,
 46437,
 46821,
 47301,
 47564,
 48099,
 50036,
 54074,
 54075,
 55483,
 55656,
 56725,
 58803,
 60456,
 60771]

In [20]:
[node[0] for node in list(graph.adjacency())]

[1269916756719685632,
 1264846057906806786,
 1277976913743503365,
 23719684,
 1285714950896443395,
 1081973336010309632,
 1284209535561818112,
 190751291,
 1279563405536325632,
 121639467,
 1279951677320093696,
 47304797,
 1286366488987865088,
 2278088432,
 1270505069826490368,
 106557003,
 1268693398854270976,
 2786302676,
 1276247727404269569,
 344264843,
 1276356681044115456,
 3266550512,
 1260354169384378368,
 964549422645424128,
 1273005746066649088,
 756328737864454144,
 1267241848747196419,
 132800841,
 1267234008250925057,
 3224525778,
 1278093777836572673,
 566393365,
 1276636801457778688,
 2297338710,
 1278331259022766080,
 376521614,
 1283233478595817474,
 73814427,
 1284160746134405123,
 509626471,
 1274308986230226946,
 1000799129898012673,
 1282129445109800962,
 125805827,
 1283761946873880577,
 1124022135335325696,
 1275707453011869696,
 22657914,
 1277723360114475008,
 1303552081,
 1258420248597852162,
 316418293,
 1267606847072198658,
 725408525925515264,
 126724033388

In [36]:

[node[0] for node in list(graph.nodes())].index(751661942)

TypeError: 'int' object is not subscriptable

In [72]:
torch.sort(torch.sigmoid(output * output[3]))

torch.return_types.sort(
values=tensor([0.5000, 0.5000, 0.5000,  ..., 1.0000, 1.0000, 1.0000]),
indices=tensor([38833, 38820, 38821,  ..., 37791, 37793, 37792]))

In [69]:
output[3]

tensor(0.0042)

tensor(0.0008)

In [54]:
probs = torch.sigmoid(output)
sources = G.edge_label_index[0]
targets = G.edge_label_index[1]

target_node_id = 0 # Example target node ID
# Find edges where either source or target is the target_node_id
mask = (sources == target_node_id) | (targets == target_node_id)

# Get the probabilities and edge indices for that node
node_probs = probs[mask]
node_edges = G.edge_label_index[:, mask]  # 
node_probs

tensor([0.5002, 0.5002])

In [12]:
G.node_label_index

tensor([    0,     1,     2,  ..., 60890, 60891, 60892])

In [5]:
G.node_label

tensor([1, 3, 1,  ..., 1, 1, 1])

In [20]:
G.node_feature[0]

tensor([0., 0., 0., 0., 1.])

In [5]:
data.to_pandas()

,bookmark_count,favorite_count,favourites_count,follower_count,following_count,friends,is_blue_verified,is_hateful,listed_count,number_of_tweets,quote_count,reply_count,retweet_count,source,tweet_id,user_id,views
0,0,0,941,39,6,6,False,1,0,422,0,0,0,0,1269916756719685632,1264846057906806786,0
1,23,4860,13269,95956,1596,1595,True,1,245,21715,28,334,156,0,1277976913743503365,23719684,0
2,0,2,1494,1394,427,427,False,1,30,1173,0,0,0,0,1285714950896443395,1081973336010309632,0
3,0,4,2675,49,165,165,False,1,3,1866,0,1,0,0,1284209535561818112,190751291,0
4,0,2,6,214664,258,258,False,1,0,310173,1,7,1,0,1279563405536325632,121639467,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35912,0,1,230,1182,30,30,False,1,0,2480,0,0,0,0,1275176503236612097,708865524751572992,0
35913,0,2,411,42,83,83,False,1,0,256,0,0,1,0,1272830576140136448,2189107029,0
35914,2,64,19738,13636,453,453,False,1,102,4005,1,0,9,0,1266939509251473409,271747129,0
35915,0,0,1524,115,42,41,False,1,1,506,0,0,0,0,1274516570606272512,1266165406370279428,0
